# Rủi ro tín dụng Freddie Mac vintage 2019
Notebook đọc ba mart trong PostgreSQL `analytics.serving`, không đọc dữ liệu khoản vay cá nhân. Chạy `make run-all` trước, rồi `make notebook`.

Tỷ lệ theo tháng dùng số khoản vay đang hoạt động; loại `XX` khỏi mẫu số quá hạn. Không cộng dư nợ qua nhiều tháng. Tên bên bán/servicer có nhóm `OTHER`; nhóm này không đại diện một tổ chức duy nhất. Gold giữ nhóm điểm/LTV/DTI theo phiên bản tại tháng báo cáo; CORR1 không viết lại lịch sử các tháng cũ.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from IPython.display import display
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'conf/pipeline.yml').is_file())
env = {}
for line in (root / '.env').read_text().splitlines():
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        env[key.strip()] = value.strip()

NOTEBOOK_DATASET = "sample"
NOTEBOOK_PG_PORT = "5432"

def query(sql):
    with psycopg2.connect(host=os.environ.get('NOTEBOOK_PG_HOST', '127.0.0.1'),
                         port=NOTEBOOK_PG_PORT, dbname='analytics',
                         user=env['POSTGRES_USER'], password=env['POSTGRES_PASSWORD']) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

publication = query("SELECT dataset, ready FROM control.gold_publication")
assert publication.loc[publication.dataset == NOTEBOOK_DATASET, 'ready'].any(), 'Chưa có Gold hợp lệ cho dataset này'
d = query('SELECT * FROM serving.mart_delinquency')
p = query('SELECT * FROM serving.mart_prepayment ORDER BY reporting_date')
c = query('SELECT * FROM serving.mart_covid_forbearance')
assert not d.empty and not p.empty
for frame in (d, p, c):
    if 'reporting_date' in frame:
        frame['reporting_date'] = pd.to_datetime(frame['reporting_date'])
    for column in frame:
        if column.endswith(('_upb', '_rate')) or column in ('smm', 'cpr'):
            frame[column] = pd.to_numeric(frame[column])

print("Nguồn:", NOTEBOOK_DATASET, "| Số tháng đã nạp:", p.reporting_date.nunique(),
      "| Từ", p.reporting_date.min().date(), "đến", p.reporting_date.max().date())

def plot_rates(group, title):
    sums = d.groupby(group)[['eligible_count', 'd30_count', 'd60_count', 'd90_plus_count']].sum()
    rates = sums[['d30_count', 'd60_count', 'd90_plus_count']].div(sums.eligible_count.replace(0, float('nan')), axis=0)
    ax = rates.plot(figsize=(11, 4), title=title)
    ax.set_ylabel('Tỷ lệ khoản vay'); plt.tight_layout(); plt.show()
    return rates


## 1. Quá hạn thay đổi thế nào, nhất là COVID 2020–2021?
Biểu đồ thể hiện ba nhóm loại trừ nhau: 30–59, 60–89 và 90+ ngày. Phải cộng tử số/mẫu số trước khi chia, không lấy trung bình các tỷ lệ của từng nhóm.

In [ ]:
monthly = plot_rates('reporting_date', 'Quá hạn theo tháng báo cáo')
display(monthly.loc['2020-01-01':'2021-12-01'].agg(['mean', 'max']))

## 2. Điểm tín dụng, LTV và DTI liên hệ với quá hạn ra sao?
Đây là tương quan mô tả, không chứng minh quan hệ nhân quả. Biểu đồ gộp các tháng nên mẫu số là số lần quan sát khoản vay–tháng.

In [ ]:
for band in ('credit_score_band', 'ltv_band', 'dti_band'):
    display(plot_rates(band, 'Quá hạn theo ' + band))

## 3. Bang và bên bán nào có tỷ lệ 90+ cao?
Hiện 10 nhóm có tỷ lệ cao nhất và mẫu số ít nhất 100 khoản vay–tháng để hạn chế nhóm quá nhỏ.

In [ ]:
for group in ('property_state', 'seller_name'):
    sums = d.groupby(group)[['eligible_count', 'd90_plus_count']].sum()
    sums['rate'] = sums.d90_plus_count / sums.eligible_count.replace(0, float('nan'))
    top = sums[sums.eligible_count >= 100].nlargest(10, 'rate')
    top.rate.plot.barh(title='Tỷ lệ 90+ theo ' + group, figsize=(10, 4)); plt.tight_layout(); plt.show()
    display(top)

## 4. Hoãn trả COVID và kết cục sau đó
Cohort có ít nhất một tháng F trong 202003–202112. Xét 12 tháng sau tháng F cuối cùng; ưu tiên vỡ nợ, trả trước, sửa đổi; sau đó hồi phục nếu đúng hạn ở cuối cửa sổ. `UNKNOWN` biểu thị không đủ quan sát để kết luận. Đây là phân tích kết cục dùng dữ liệu tương lai, không phải đặc trưng dự báo tại thời điểm hoãn trả.

In [ ]:
if c.empty:
    print('Chưa có quan sát cohort COVID trong các kỳ đã nạp.')
else:
    outcomes = c.groupby('outcome').loan_count.sum()
    outcomes.plot.bar(title='Kết cục sau hoãn trả COVID', figsize=(9, 4)); plt.tight_layout(); plt.show()
    display(outcomes.to_frame()); print('Tổng cohort:', int(outcomes.sum()))


## 5. Tỷ lệ trả trước hạn thay đổi thế nào?
SMM = dư nợ xóa sổ do trả trước tháng m / dư nợ hoạt động tháng m−1; CPR = 1 − (1 − SMM)^12. Tháng đầu thiếu mẫu số được giữ null. Dữ liệu không có lý do trả trước; không thể khẳng định một khoản vay trả trước do tái cấp vốn.

In [ ]:
p.set_index('reporting_date')[['smm', 'cpr']].plot(title='Trả trước hạn: SMM và CPR', figsize=(11, 4))
plt.tight_layout(); plt.show()
display(p[['reporting_date', 'prepaid_count', 'smm', 'cpr']].tail(12))